# SpaceJEPA v3 · 2. Fly the study

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rk0604/ReEntryAI/blob/jepa-v3/StochasticEntrySim/ReEntryAI_JEPA_v3_2_fly.ipynb)

Every flight is one job keyed by *(controller, load limit, seed, MPPI repeat, case, atmosphere)*. Its
result (`04_flights/results/…json`) and its 1 Hz telemetry with RCS fuel and jet-on time
(`04_flights/telemetry/…npz`) are written to Drive the moment it lands, so a re-run skips finished flights.

| Controller | Seeds | Load limit | Flights |
|---|---|---|---|
| classical predictor-corrector | deterministic | its own guidance | 256 |
| v1 (fair budget), v2 ensemble, v2 single member | 0–4 | **4.5 g** | 3 × 1,280 |
| v1, v2 legacy | 0 | 8 g | 512 |
| v2, two extra MPPI noise streams | 0 | 4.5 g | 512 |

The learned controllers all fly through one lockstep MPPI runner, sharing env setup, planner code, cost
terms, and timing. At each 5 s decision, every flight's 256 candidate plans go through the model in one
GPU batch. A given (seed, case, atmosphere) gets **the same MPPI noise stream for every learned
controller** (common random numbers), so paired comparisons differ only in the model.

**What limits the speed is CPU cores, not the GPU.** The simulator steps on the CPU. The classical
controller is about 6 CPU-hours for 256 flights, and the learned flights about 8 CPU-hours of
simulation. Standard Colab GPU runtimes often have only 2 vCPUs. Each stage prints its measured rate
and an ETA, and you can stop and resume at any time. The classical stage can also run on any machine
with more cores, writing into the same Drive folder.

## 0. Setup

On Colab this clones the `jepa-v3` branch, installs `pymsis`, and mounts Google Drive. Every output
goes to **`MyDrive/ReEntryAI_jepa_v3/`** (`ReEntryAI_jepa_v3_smoke/` in smoke mode); the layout is in
`jepa/v3_plan.md` §9 and in the auto-written `00_study/README.md`. The 6,000 flights are read from
`MyDrive/ReEntryAI_jepa_data/wide_6000/`. Every stage is resumable: after a disconnect, run all
cells again and finished work is skipped.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
BRANCH = "jepa-v3"
MODE = os.environ.get("JEPA_V3_MODE", "full")     # "full": the study | "smoke": every stage, tiny, numbers meaningless
if IN_COLAB:
    REPO = "/content/ReEntryAI"
    if not os.path.exists(REPO):
        !git clone -q -b {BRANCH} https://github.com/rk0604/ReEntryAI.git {REPO}
    else:
        !git -C {REPO} pull -q
    %pip install -q pymsis
    os.chdir(f"{REPO}/StochasticEntrySim")
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
    STUDY_ROOT = f"{DRIVE}/ReEntryAI_jepa_v3" + ("" if MODE == "full" else f"_{MODE}")
    DATA_CANDIDATES = [f"{DRIVE}/ReEntryAI_jepa_data/wide_6000/episodes_wide_6000.pkl",
                       f"{DRIVE}/ReEntryAI_jepa_data/wide_6000"]
    LOCAL_DATA = "/content/data"
else:
    STUDY_ROOT = os.environ.get("JEPA_V3_ROOT", f"jepa_runs/v3_{MODE}")
    DATA_CANDIDATES = [os.environ.get("JEPA_WIDE_DATA", ""), "jepa_runs/episodes_wide_6000.pkl", "jepa_runs/wide_6000"]
    LOCAL_DATA = "jepa_runs"

import time
from pathlib import Path
SIM_DIR = Path.cwd()
assert (SIM_DIR / "rl_env.py").exists(), "start Jupyter in StochasticEntrySim/"
sys.path.insert(0, str(SIM_DIR))
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import Image, display
%matplotlib inline
from jepa import testset_v3 as TS, study_v3 as S, fly_v3 as F, analyze_v3 as A

CFG = S.StudyConfig.smoke() if MODE == "smoke" else S.StudyConfig()
L = S.Layout(STUDY_ROOT).make()
FROZEN = S.freeze_config(CFG, L)
CPUS = os.cpu_count() or 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"MODE={MODE}  study folder={STUDY_ROOT}  device={DEVICE}  cpus={CPUS}  git={FROZEN['git_commit'][:10]}")

In [ ]:
JOBS = S.build_jobs(CFG, L)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # inherited by the GPU workers
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == "cuda" else 0.0
CPU_WORKERS = CPUS                         # classical: one simulator per core
# learned: processes sharing the GPU, each stepping its own simulators on one core. Budget 6 GB of GPU
# memory per worker (measured peak of one rollout batch: <= 5 GB), keeping 2 GB spare: L4 -> 3, T4 -> 2.
# If a worker still runs out of GPU memory, lower GPU_WORKERS and re-run: finished flights are kept.
GPU_WORKERS = max(1, min(4, CPUS, int((GPU_MEM_GB - 2) // 6) if DEVICE == "cuda" else CPUS))
RUN_CLASSICAL, RUN_LEARNED = True, True
tab = pd.DataFrame([{"controller": j["controller"], "load": j["load"], "seed": j["seed"], "mppi_rep": j["mppi_rep"],
                     "done": S.flight_done(L, j)} for j in JOBS])
display(tab.groupby(["controller", "load", "seed", "mppi_rep"], dropna=False).done.agg(["sum", "size"])
        .rename(columns={"sum": "done", "size": "flights"}))
print(f"{len(JOBS)} flights, {int(tab.done.sum())} done. MPPI horizon {S.horizon_steps(CFG)} steps. "
      f"CPU workers {CPU_WORKERS}, GPU workers {GPU_WORKERS} on {DEVICE}.")

## 1. Classical predictor-corrector (CPU)

All 256 conditions run again, including the 64 bridge flights that v1/v2 cached, for two reasons:
every classical flight then has telemetry (the cached results had none), and the cache came from an
earlier simulator version. The guidance is deterministic: re-running gives identical flights.

In [ ]:
if RUN_CLASSICAL:
    S.run_classical(JOBS, L, workers=CPU_WORKERS)

## 2. Learned controllers (GPU + CPUs), seed by seed

Seed 0 runs completely (all controllers and arms), then seed 1, and so on. If the study stops early,
the finished seeds are complete paired comparisons. A model run whose checkpoint is not trained yet is
skipped with a note.

In [ ]:
if RUN_LEARNED:
    S.run_learned(CFG, JOBS, L, device=DEVICE, workers=GPU_WORKERS)

## 3. Merge and a first look

In [ ]:
df = S.collect_flights(L)
print(f"{len(df)} flights in {L.flights / 'flights.csv'}")
if len(df):
    df["arm"] = df.controller + " " + df.load.astype(str)
    display(df[df.mppi_rep.fillna(0) == 0].groupby(["arm", "atmosphere"]).agg(
        n=("miss_km", "size"), success5=("success_5km", "mean"), median_km=("miss_km", "median"),
        p90_km=("miss_km", lambda s: s.quantile(0.9)), peak_g=("peak_g", "mean"), fuel_kg=("rcs_fuel_kg", "mean"))
        .round(3).unstack("atmosphere"))
S.update_manifest(L);